# 🫒 Atlas-Oliva : Entraînement Intensif Hybride (Drone Zenodo + Satellite Zoom 19)
### Architecture Finale Validée — Précision Maximale & Zéro Perte de Poids

1. **Dataset Hybride Multi-Échelles :** Toutes les images Drone réelles (**Zenodo `13121962`**, ~309 Mo) + tuiles **Satellite Zoom 19** (Haouz Marrakech), partitionnées en **70% Train / 15% Validation / 15% Test** (`seed=42`).
2. **Modèle 1 (Détection & Centroïdes) :** **`YOLO11s`** à **`imgsz=800` + `multi_scale=True`**, `batch=12`, **`120 époques`** (`patience=25`), `AdamW`, `cos_lr=True`.
3. **Modèle 2 (Santé : `sain`, `stress`, `suspect`) :** **`YOLO11s-cls`** avec **rééquilibrage strict uniquement dans Train** (Val et Test **100% réels sans fuite de données**), **`60 époques`** (`patience=15`), `batch=64`.
4. **Sauvegarde Automatique Google Drive :** Tous les checkpoints (`best.pt`) sont écrits directement dans `/content/drive/MyDrive/Atlas_Oliva_Models/`.

**Supervisé par :** Pr. Mohamed Lachgar (*EMSI & Université Cadi Ayyad*)  
**Auteurs :** Moad Moussaoui & Noussaire Benassou

## ⚙️ 1. Montage Google Drive, Vérification GPU Tesla T4 & Dépendances

In [ ]:
import os, sys, shutil, zipfile, random, math, urllib.request
import torch
import numpy as np
import cv2
from pathlib import Path
from google.colab import drive

# 1. Montage Google Drive pour sauvegarde permanente des modèles
try:
    drive.mount('/content/drive')
    DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/Atlas_Oliva_Models")
except Exception as e:
    print("⚠️ Google Drive non monté, sauvegarde locale dans /content/Atlas_Oliva_Models :", e)
    DRIVE_SAVE_DIR = Path("/content/Atlas_Oliva_Models")
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("💾 Sauvegarde Google Drive active :", DRIVE_SAVE_DIR)

# 2. Vérification GPU
print("Python :", sys.version.split()[0], "| PyTorch :", torch.__version__)
if torch.cuda.is_available():
    print("GPU Détecté :", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    raise RuntimeError("⚠️ Activez le GPU : Exécution > Modifier le type d'exécution > T4 GPU")

!pip install -q ultralytics opencv-python matplotlib seaborn pandas

## 📥 2. Téléchargement Direct du Dataset Hybride (Drone Zenodo `13121962` + Satellite Zoom 19) & Split 70/15/15
- Télécharge les 3 archives Drone réelles directement dans Colab (**0 Mo sur votre PC**).
- Ajoute les tuiles Satellite Zoom 19 (Marrakech Haouz) pour rendre le modèle robuste aux deux échelles (Drone + Carte Leaflet).
- Mélange (`seed=42`) et coupe en **70% Train / 15% Validation / 15% Test**.

In [ ]:
import os, shutil, zipfile, random, math, urllib.request
import numpy as np
import cv2
from pathlib import Path

WORK_DIR = Path("/content/olive_hybrid_pipeline")
RAW_DIR = WORK_DIR / "raw_zenodo"
SAT_DIR = WORK_DIR / "raw_satellite"
DETECTION_DIR = WORK_DIR / "dataset_detection"
HEALTH_DIR = WORK_DIR / "dataset_health"

for d in [DETECTION_DIR, HEALTH_DIR]:
    if d.exists(): shutil.rmtree(d)
RAW_DIR.mkdir(parents=True, exist_ok=True)
SAT_DIR.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    (DETECTION_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DETECTION_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    for cls in ["sain", "stress", "suspect"]:
        (HEALTH_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# --- A. TÉLÉCHARGEMENT DIRECT DU DATASET HYBRIDE VÉRIFIÉ (140 images réelles, 10 931 oliviers) ---
# (Zenodo bloquant wget par 403 Forbidden sur Colab, l'archive vérifiée est hébergée directement sur votre dépôt GitHub)
hybrid_zip_url = "https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/datasets/atlas_oliva_hybrid_dataset.zip"
zpath = RAW_DIR / "atlas_oliva_hybrid_dataset.zip"
if not zpath.exists() or not zipfile.is_zipfile(zpath):
    print("⬇️ Téléchargement du dataset hybride (Drone + Satellite, 10 931 oliviers)...")
    urllib.request.urlretrieve(hybrid_zip_url, str(zpath))

assert zipfile.is_zipfile(zpath), "❌ Le fichier téléchargé n'est pas une archive ZIP valide !"
extract_folder = RAW_DIR / "atlas_oliva_hybrid"
if extract_folder.exists(): shutil.rmtree(extract_folder)
with zipfile.ZipFile(zpath, 'r') as zf:
    zf.extractall(extract_folder)
print(f"✅ Archive extraite avec succès ({zpath.stat().st_size / (1024*1024):.2f} Mo)")

# Collecte des paires (image, label)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
all_drone_imgs = [p for p in (extract_folder / "images").glob("*") if p.suffix.lower() in IMG_EXTS]
paired_data = []
for img_p in sorted(all_drone_imgs):
    lbl_p = extract_folder / "labels" / f"{img_p.stem}.txt"
    if lbl_p.exists() and lbl_p.stat().st_size > 0:
        paired_data.append((img_p, lbl_p))

print(f"✅ Images réelles annotées collectées (Drone + Satellite) : {len(paired_data)}")

# --- B. AJOUT DES TUILES SATELLITE ZOOM 19 (HYBRIDE LEAFLET) ---
sat_urls = [
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/screenshot_zoom19_full.png", "sat_haouz_1.png"),
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/left_parcel_calibrated.jpg", "sat_haouz_2.jpg"),
    ("https://raw.githubusercontent.com/moadms/OliveSmart-Project/main/resultats_tests/detail_trees_v2.jpg", "sat_haouz_3.jpg")
]
sat_tiles_dir = SAT_DIR / "tiles"
sat_tiles_dir.mkdir(exist_ok=True)
sat_count = 0

for url, name in sat_urls:
    dest = SAT_DIR / name
    try:
        if not dest.exists():
            urllib.request.urlretrieve(url, str(dest))
        img = cv2.imread(str(dest))
        if img is None: continue
        H, W, _ = img.shape
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        R, G, B = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
        lum = R + G + B + 1e-6
        foliage = (G > R * 0.92) & (G > B * 0.92) & (lum > 55) & (lum < 580)
        clean = cv2.morphologyEx(np.uint8(foliage * 255), cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
        dist = cv2.GaussianBlur(cv2.distanceTransform(clean, cv2.DIST_L2, 5), (5, 5), 0)
        peaks = (dist == cv2.dilate(dist, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))) & (dist >= 2.5)
        num_p, _, _, centroids = cv2.connectedComponentsWithStats(np.uint8(peaks * 255))
        trees = sorted([(centroids[i][0], centroids[i][1], float(dist[int(centroids[i][1]), int(centroids[i][0])])) for i in range(1, num_p)], key=lambda t: -t[2])
        kept = []
        for cx, cy, dv in trees:
            if all(math.hypot(cx - kx, cy - ky) >= 11.0 for kx, ky, _ in kept):
                kept.append((cx, cy, max(9, min(26, int(round(dv * 1.35))))))
        ts, step = 640, 200
        for y0 in range(0, max(1, H - ts + 1), step):
            for x0 in range(0, max(1, W - ts + 1), step):
                sub = img[y0:y0+ts, x0:x0+ts]
                if sub.shape[0] != ts or sub.shape[1] != ts:
                    pad = np.zeros((ts, ts, 3), dtype=np.uint8)
                    pad[:sub.shape[0], :sub.shape[1]] = sub
                    sub = pad
                lbls = [f"0 {(cx-x0)/ts:.6f} {(cy-y0)/ts:.6f} {(r*2)/ts:.6f} {(r*2)/ts:.6f}" for cx, cy, r in kept if x0 <= cx < x0+ts and y0 <= cy < y0+ts]
                if lbls:
                    ip = sat_tiles_dir / f"sat_tile_{sat_count:04d}.jpg"
                    lp = sat_tiles_dir / f"sat_tile_{sat_count:04d}.txt"
                    cv2.imwrite(str(ip), sub)
                    lp.write_text("\n".join(lbls))
                    paired_data.append((ip, lp))
                    sat_count += 1
    except Exception as e:
        print(f"⚠️ Satellite {name} ignoré : {e}")

print(f"✅ Tuiles Satellite Zoom 19 ajoutées : {sat_count}")
print(f"🌍 TOTAL HYBRIDE (Drone + Satellite) : {len(paired_data)} images")

# --- C. SPLIT STRICT 70% TRAIN / 15% VALIDATION / 15% TEST ---
random.seed(42)
random.shuffle(paired_data)
n_total = len(paired_data)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

splits = {
    "train": paired_data[:n_train],
    "val": paired_data[n_train:n_train + n_val],
    "test": paired_data[n_train + n_val:]
}

total_boxes = 0
for split_name, items in splits.items():
    for idx, (img_p, lbl_p) in enumerate(items):
        dst_img = DETECTION_DIR / split_name / "images" / f"{split_name}_{idx:05d}{img_p.suffix.lower()}"
        dst_lbl = DETECTION_DIR / split_name / "labels" / f"{split_name}_{idx:05d}.txt"
        shutil.copy2(img_p, dst_img)
        clean_lines = []
        with open(lbl_p, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = map(float, parts[1:5])
                    if 0 < bw <= 1.0 and 0 < bh <= 1.0 and 0 <= xc <= 1.0 and 0 <= yc <= 1.0:
                        clean_lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
                        total_boxes += 1
        dst_lbl.write_text("\n".join(clean_lines))

(DETECTION_DIR / "data.yaml").write_text(
    f"path: {DETECTION_DIR.as_posix()}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 1\nnames:\n  0: Olive_Tree\n"
)

print("=" * 65)
print(f"🎉 DATASET HYBRIDE PRÊT ({total_boxes} oliviers annotés) :")
print(f"   • 📦 Train (70%)      : {len(splits['train'])} images")
print(f"   • 📦 Validation (15%) : {len(splits['val'])} images")
print(f"   • 📦 Test (15%)       : {len(splits['test'])} images")
print("=" * 65)

## 🏋️ 3. Entraînement Intensif MODÈLE 1 : `YOLO11s` (`imgsz=800`, `multi_scale=True`, `120 époques`)
- Sauvegardé directement sur **Google Drive** (`/MyDrive/Atlas_Oliva_Models/runs_detector/`).
- `patience=25` (Early Stopping automatique si le modèle atteint son pic avant 120 époques).
- `close_mosaic=15` : désactive le mosaic sur les 15 dernières époques pour affiner la précision des centroïdes.

In [ ]:
from ultralytics import YOLO

detector_model = YOLO("yolo11s.pt")

results_det = detector_model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=120,
    patience=25,
    imgsz=800,
    multi_scale=True,
    batch=12,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    plots=True,
    save=True,
    project=str(DRIVE_SAVE_DIR / "runs_detector"),
    name="yolo11s_hybrid_800_120ep",
    exist_ok=True
)

# Évaluation impartiale sur le TEST SET (15% jamais vus)
best_det_path = DRIVE_SAVE_DIR / "runs_detector/yolo11s_hybrid_800_120ep/weights/best.pt"
shutil.copy2(best_det_path, DRIVE_SAVE_DIR / "best_yolo11s_detector.pt")

best_detector = YOLO(str(best_det_path))
test_metrics = best_detector.val(data=str(DETECTION_DIR / "data.yaml"), split="test", imgsz=800, plots=True)

print("\n" + "=" * 65)
print("📊 RÉSULTATS FINAUX MODÈLE 1 (TEST SET 15% — HYBRID 800px) :")
print(f"   • 🏆 mAP@50      : {test_metrics.box.map50 * 100:.2f}%")
print(f"   • 🏆 mAP@50-95   : {test_metrics.box.map * 100:.2f}%")
print(f"   • 🎯 Précision   : {test_metrics.box.mp * 100:.2f}%")
print(f"   • 🎯 Rappel      : {test_metrics.box.mr * 100:.2f}%")
print(f"   • 💾 Sauvegardé  : {DRIVE_SAVE_DIR / 'best_yolo11s_detector.pt'}")
print("=" * 65)

## 🌿 4. Dataset Santé (`sain`, `stress`, `suspect`) : Split Propre 70/15/15 + Rééquilibrage Strict dans Train
- **Zéro fuite de données (Data Leakage = 0%) :** Les couronnes issues des images `train` vont uniquement dans `train`, celles de `val` dans `val`, et celles de `test` dans `test`.
- **Rééquilibrage uniquement dans `train` :** Les classes minoritaires (`stress`, `suspect`) sont augmentées par rotations (90°, 180°, 270°) et flips pour égaler la classe `sain`.
- **`val` et `test` restent 100% réels et non dupliqués** pour une évaluation scientifique honnête.

In [ ]:
MAX_SAIN_TRAIN = 2000
crops_split = {sp: {"sain": [], "stress": [], "suspect": []} for sp in ["train", "val", "test"]}

for sp in ["train", "val", "test"]:
    for img_p in (DETECTION_DIR / sp / "images").glob("*"):
        lbl_p = DETECTION_DIR / sp / "labels" / f"{img_p.stem}.txt"
        if not lbl_p.exists(): continue
        img = cv2.imread(str(img_p))
        if img is None: continue
        H, W, _ = img.shape
        with open(lbl_p, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = max(0, int((xc - bw/2)*W)), max(0, int((yc - bh/2)*H))
                x2, y2 = min(W, int((xc + bw/2)*W)), min(H, int((yc + bh/2)*H))
                crop = img[y1:y2, x1:x2]
                if crop.shape[0] < 16 or crop.shape[1] < 16: continue
                ch, cw, _ = crop.shape
                core = crop[ch//4:3*ch//4, cw//4:3*cw//4]
                if core.size == 0: core = crop
                rgb = cv2.cvtColor(core, cv2.COLOR_BGR2RGB).astype(np.float32)
                R, G, B = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
                exg = float(np.mean(2.0 * G - R - B))
                gcc = float(np.mean(G / (R + G + B + 1e-6)))
                cls = "sain" if (exg >= 9.0 and gcc >= 0.352) else ("stress" if (exg >= 2.0 and gcc >= 0.320) else "suspect")
                if cls == "sain" and sp == "train" and len(crops_split[sp][cls]) >= MAX_SAIN_TRAIN:
                    continue
                crops_split[sp][cls].append(cv2.resize(crop, (128, 128)))

print("📊 Répartition BRUTE avant rééquilibrage Train :")
for sp in ["train", "val", "test"]:
    print(f"   • {sp.upper():5s} :", {k: len(v) for k, v in crops_split[sp].items()})

# Rééquilibrage STRICT uniquement dans TRAIN (rotations 90/180/270 + flips)
def augment_crop(img_arr, variant):
    v = variant % 7
    if v == 0: return cv2.flip(img_arr, 1)
    if v == 1: return cv2.flip(img_arr, 0)
    if v == 2: return cv2.rotate(img_arr, cv2.ROTATE_90_CLOCKWISE)
    if v == 3: return cv2.rotate(img_arr, cv2.ROTATE_180)
    if v == 4: return cv2.rotate(img_arr, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if v == 5: return cv2.convertScaleAbs(cv2.flip(img_arr, 1), alpha=1.08, beta=4)
    return cv2.convertScaleAbs(cv2.rotate(img_arr, cv2.ROTATE_90_CLOCKWISE), alpha=0.92, beta=-4)

target_train = max(len(v) for v in crops_split["train"].values())
for cls in ["sain", "stress", "suspect"]:
    base = crops_split["train"][cls]
    if not base: continue
    orig_len = len(base)
    idx = 0
    while len(base) < target_train:
        base.append(augment_crop(base[idx % orig_len], idx))
        idx += 1

# Sauvegarde sur disque (Val et Test restent 100% réels !)
for sp in ["train", "val", "test"]:
    for cls in ["sain", "stress", "suspect"]:
        for i, c in enumerate(crops_split[sp][cls]):
            cv2.imwrite(str(HEALTH_DIR / sp / cls / f"{cls}_{i:05d}.jpg"), c)

print("\n✅ Répartition FINALE après rééquilibrage Train (Val/Test 100% réels) :")
for sp in ["train", "val", "test"]:
    print(f"   • {sp.upper():5s} :", {k: len(list((HEALTH_DIR / sp / k).glob('*.jpg'))) for k in ["sain", "stress", "suspect"]})

## 🩺 5. Entraînement MODÈLE 2 : Diagnostic Sanitaire (`YOLO11s-cls` — 60 Époques) & Sauvegarde Drive

In [ ]:
health_model = YOLO("yolo11s-cls.pt")

results_health = health_model.train(
    data=str(HEALTH_DIR),
    epochs=60,
    patience=15,
    imgsz=128,
    batch=64,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    fliplr=0.5,
    flipud=0.5,
    plots=True,
    project=str(DRIVE_SAVE_DIR / "runs_health"),
    name="yolo11s_health_balanced_60ep",
    exist_ok=True
)

best_health_path = DRIVE_SAVE_DIR / "runs_health/yolo11s_health_balanced_60ep/weights/best.pt"
shutil.copy2(best_health_path, DRIVE_SAVE_DIR / "best_yolo11s_health.pt")

best_health = YOLO(str(best_health_path))
health_test = best_health.val(data=str(HEALTH_DIR), split="test", plots=True)

print("\n" + "=" * 65)
print(f"🏆 Exactitude Top-1 sur TEST SET 100% Réel : {health_test.top1 * 100:.2f}%")
print(f"💾 Modèle 1 sauvegardé sur Drive : {DRIVE_SAVE_DIR / 'best_yolo11s_detector.pt'}")
print(f"💾 Modèle 2 sauvegardé sur Drive : {DRIVE_SAVE_DIR / 'best_yolo11s_health.pt'}")
print("=" * 65)